##### 🥈 SILVER LAYER
Data Cleaning, Schema Enforcement and Null Handling and Saving as Delta Lake

In [ ]:
#Load environment variables from .env files 
from dotenv import load_dotenv
import os

load_dotenv("coding.env")

In [4]:
#Initiate Spark Session

import os
import sys

os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable
os.environ["HADOOP_HOME"] = "C:\\hadoop"
os.environ["hadoop.home.dir"] = "C:\\hadoop"
sys.path.append("C:\\hadoop\\bin")

from pyspark.sql import SparkSession


from delta import configure_spark_with_delta_pip

builder = SparkSession.builder \
    .appName("PyProject") \
    .master("local[*]") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.jars.packages", "io.delta:delta-spark_2.13:4.1.0")

spark = configure_spark_with_delta_pip(builder).getOrCreate()
spark

In [ ]:
#Reading parquet file from BRONZE LAYER to a new dataframe

path = os.getenv("file_path1")
sales = spark.read.parquet(path)


In [7]:
#Call an action to show the Sales data
sales.show()

+--------------+-----------+--------------------+------------+--------------+--------+-----------+--------------+--------+----------------+----------------+--------------------+
|Transaction ID|Customer ID|            Category|        Item|Price Per Unit|Quantity|Total Spent|Payment Method|Location|Transaction Date|Discount Applied|        datetime_now|
+--------------+-----------+--------------------+------------+--------------+--------+-----------+--------------+--------+----------------+----------------+--------------------+
|   TXN_6867343|    CUST_09|          Patisserie| Item_10_PAT|          18.5|    10.0|      185.0|Digital Wallet|  Online|      2024-04-08|            True|2026-02-27 14:12:...|
|   TXN_3731986|    CUST_22|       Milk Products|Item_17_MILK|          29.0|     9.0|      261.0|Digital Wallet|  Online|      2023-07-23|            True|2026-02-27 14:12:...|
|   TXN_9303719|    CUST_02|            Butchers| Item_12_BUT|          21.5|     2.0|       43.0|   Credit Ca

In [8]:
#Check the schema
sales.printSchema()

root
 |-- Transaction ID: string (nullable = true)
 |-- Customer ID: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Item: string (nullable = true)
 |-- Price Per Unit: string (nullable = true)
 |-- Quantity: string (nullable = true)
 |-- Total Spent: string (nullable = true)
 |-- Payment Method: string (nullable = true)
 |-- Location: string (nullable = true)
 |-- Transaction Date: string (nullable = true)
 |-- Discount Applied: string (nullable = true)
 |-- datetime_now: timestamp (nullable = true)



In [ ]:
#Defining Schema for SILVER LAYER

from pyspark.sql.types import StructType, StructField,StringType, DateType, TimestampType, FloatType

schema_sales = StructType([StructField("Transaction ID", StringType(), True),
                        StructField("Customer ID", StringType(), True),
                        StructField("Category", StringType(), True),
                        StructField("Item", StringType(), True),
                        StructField("Price Per Unit", FloatType(), True),
                        StructField("Quantity",FloatType(), True),
                        StructField("Total Spent", FloatType(), True),
                        StructField("Payment Method", StringType(), True),
                        StructField("Location", StringType(), True),
                        StructField("Transaction Date",DateType(), True),
                        StructField("Discount Applied", StringType(), True),
                        StructField("datetime_now", TimestampType(), True)
                        ])


In [ ]:
#Create Dataframe for SILVER LAYER

from pyspark.sql.functions import col
sales_silver = sales 
for field in schema_sales.fields:
    sales_silver = sales_silver.withColumn(field.name, col(field.name).cast(field.dataType))



In [11]:
#Check the schema
sales_silver.printSchema()

root
 |-- Transaction ID: string (nullable = true)
 |-- Customer ID: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Item: string (nullable = true)
 |-- Price Per Unit: float (nullable = true)
 |-- Quantity: float (nullable = true)
 |-- Total Spent: float (nullable = true)
 |-- Payment Method: string (nullable = true)
 |-- Location: string (nullable = true)
 |-- Transaction Date: date (nullable = true)
 |-- Discount Applied: string (nullable = true)
 |-- datetime_now: timestamp (nullable = true)



Schema Defined

In [ ]:
#Call an action to display dataframe

sales_silver.show()

+--------------+-----------+--------------------+------------+--------------+--------+-----------+--------------+--------+----------------+----------------+--------------------+
|Transaction ID|Customer ID|            Category|        Item|Price Per Unit|Quantity|Total Spent|Payment Method|Location|Transaction Date|Discount Applied|        datetime_now|
+--------------+-----------+--------------------+------------+--------------+--------+-----------+--------------+--------+----------------+----------------+--------------------+
|   TXN_6867343|    CUST_09|          Patisserie| Item_10_PAT|          18.5|    10.0|      185.0|Digital Wallet|  Online|      2024-04-08|            True|2026-02-27 14:12:...|
|   TXN_3731986|    CUST_22|       Milk Products|Item_17_MILK|          29.0|     9.0|      261.0|Digital Wallet|  Online|      2023-07-23|            True|2026-02-27 14:12:...|
|   TXN_9303719|    CUST_02|            Butchers| Item_12_BUT|          21.5|     2.0|       43.0|   Credit Ca

In [13]:
# Cleaning and Manipulation
from pyspark.sql.functions import coalesce, lit, col

#Fix and Transform NULLs

sales_silver_null = sales_silver \
    .withColumn("Transaction ID", coalesce(col("Transaction ID"), lit("Unknown"))) \
    .withColumn("Customer ID", coalesce(col("Customer ID"), lit("Unknown"))) \
    .withColumn("Category", coalesce(col("Category"), lit("Unknown"))) \
    .withColumn("Item", coalesce(col("Item"), lit("Unknown"))) \
    .withColumn("Payment Method", coalesce(col("Payment Method"), lit("Unknown"))) \
    .withColumn("Location", coalesce(col("Location"), lit("Unknown"))) \
    .withColumn("Transaction Date", coalesce(col("Transaction Date"), lit("Unknown"))) \
    .withColumn("Discount Applied", coalesce(col("Discount Applied"), lit("Unknown")))



In [14]:
#Call an action to show the dataframe
sales_silver_null.show()

+--------------+-----------+--------------------+------------+--------------+--------+-----------+--------------+--------+----------------+----------------+--------------------+
|Transaction ID|Customer ID|            Category|        Item|Price Per Unit|Quantity|Total Spent|Payment Method|Location|Transaction Date|Discount Applied|        datetime_now|
+--------------+-----------+--------------------+------------+--------------+--------+-----------+--------------+--------+----------------+----------------+--------------------+
|   TXN_6867343|    CUST_09|          Patisserie| Item_10_PAT|          18.5|    10.0|      185.0|Digital Wallet|  Online|      2024-04-08|            True|2026-02-27 14:12:...|
|   TXN_3731986|    CUST_22|       Milk Products|Item_17_MILK|          29.0|     9.0|      261.0|Digital Wallet|  Online|      2023-07-23|            True|2026-02-27 14:12:...|
|   TXN_9303719|    CUST_02|            Butchers| Item_12_BUT|          21.5|     2.0|       43.0|   Credit Ca

In [15]:
# Removing Duplicates

sales_silver_unique = sales_silver_null.distinct() 


In [16]:
# Call an action to show the unique rows

sales_silver_unique.show()

+--------------+-----------+--------------------+------------+--------------+--------+-----------+--------------+--------+----------------+----------------+--------------------+
|Transaction ID|Customer ID|            Category|        Item|Price Per Unit|Quantity|Total Spent|Payment Method|Location|Transaction Date|Discount Applied|        datetime_now|
+--------------+-----------+--------------------+------------+--------------+--------+-----------+--------------+--------+----------------+----------------+--------------------+
|   TXN_5418871|    CUST_04|Computers and ele...| Item_22_CEA|          36.5|     7.0|      255.5|          Cash|In-store|      2022-03-16|           False|2026-02-27 14:12:...|
|   TXN_1712455|    CUST_25|            Butchers|  Item_5_BUT|          11.0|     9.0|       99.0|   Credit Card|In-store|      2024-03-11|         Unknown|2026-02-27 14:12:...|
|   TXN_6166688|    CUST_08|          Patisserie| Item_25_PAT|          41.0|     6.0|      246.0|   Credit Ca

In [17]:
# Verify Row counts

print("Before duplicates: ", sales_silver_null.count())
print("After duplicates: ", sales_silver_unique.count())

Before duplicates:  12575
After duplicates:  12575


No Duplicates Found

In [ ]:
# Rename columns

sales_silver_unique = sales_silver_unique \
    .withColumnRenamed("Transaction ID", "Transaction_ID") \
    .withColumnRenamed("Customer ID", "Customer_ID") \
    .withColumnRenamed("Price Per Unit", "Price_Per_Unit") \
    .withColumnRenamed("Total Spent", "Total_Spent") \
    .withColumnRenamed("Payment Method", "Payment_Method") \
    .withColumnRenamed("Transaction Date", "Transaction_Date") \
    .withColumnRenamed("Discount Applied", "Discount_Applied")

In [19]:
# Save as Delta Lake

silver_path = os.getenv("SILVER_PATH")
sales_silver_unique.write.mode("overwrite").format("delta").save(silver_path)

DELTA LAKE CREATED